# Trung bình dự đoán qua ba seed

Bước 13 của kế hoạch S6E11.

Giữ nguyên outer fold và chạy lại XGB-D03 với một seed mô hình khác và một seed bộ mã hóa khác. So trung bình xác suất với trung bình thứ hạng của ba lượt.

## Chạy lại

Chạy từ thư mục `plan_runs` với môi trường Python đã dùng cho RTX 4060. Trình chạy tiếp tục từ checkpoint có cấu hình khớp. Cell dưới đọc kết quả theo mặc định.

```powershell
& '..\.build\venv\Scripts\python.exe' -B -X utf8 run_plan.py --stage advanced --compare-xgb-digits --tune-family xgb --seed-anchor XGB-D03
```

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd().resolve()
roots = [Path(os.environ["S6E11_PLAN_RUNS"])] if "S6E11_PLAN_RUNS" in os.environ else []
for base in (here, *here.parents):
    roots.extend((base, base / "plan_runs", base / "s6e11_gpu" / "plan_runs",
                  base / "Predicting Credit Card Fraud" / "s6e11_gpu" / "plan_runs"))
plan_dir = next((path for path in roots if (path / "run_plan.py").is_file()), None)
if plan_dir is None:
    raise FileNotFoundError("Mở notebook trong dự án hoặc đặt S6E11_PLAN_RUNS.")

CHAY_LAI = False
if CHAY_LAI:
    subprocess.run([sys.executable, "-B", "-X", "utf8", str(plan_dir / "run_plan.py"),
                    *['--stage', 'advanced', '--compare-xgb-digits', '--tune-family', 'xgb', '--seed-anchor', 'XGB-D03']],
                   cwd=plan_dir, check=True)

## Kết quả tại thời điểm tạo notebook

| Mã | Đối chứng | ROC AUC OOF | Δ AUC | AP | Log-loss | Giây | Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 |
| :--- | :--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| `SEED3-MEAN` | `XGB-D03` | 0.950269 | +0.000087 | 0.528975 | 0.044409 | — | 0.949866 | 0.950974 | 0.955095 | 0.951069 | 0.944623 |
| `SEED3-RANK` | `XGB-D03` | 0.950263 | +0.000081 | 0.528939 | 0.931015 | — | 0.949846 | 0.950968 | 0.955090 | 0.951069 | 0.944623 |
| `XGB-D03-S93` | `XGB-D03` | 0.950091 | -0.000091 | 0.527930 | 0.044470 | 338.1 | 0.949597 | 0.950775 | 0.955045 | 0.950776 | 0.944449 |
| `XGB-D03-S71` | `XGB-D03` | 0.950074 | -0.000108 | 0.527848 | 0.044479 | 385.2 | 0.949702 | 0.950915 | 0.954834 | 0.950920 | 0.944399 |

## Đọc dữ liệu kết quả hiện tại

Cell này xác minh những lượt hoàn tất từ dự đoán OOF và tệp metric.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

sys.path.insert(0, str(plan_dir))
from report_results import final_summary, read_stage

data_dir = plan_dir / "results" / "full"
rows, pending, notes = read_stage('06-seed', data_dir)
if rows:
    points = [{"mã": row["name"], "đối chứng": row["config"].get("parent"),
               "ROC AUC OOF": row["oof_auc"], "AP": row["average_precision"],
               "delta AUC": row.get("delta_oof_auc"), "log-loss": row["log_loss"],
               **{f"fold {item['fold']}": item["auc"] for item in row["folds"]}}
              for row in rows]
    display(pd.DataFrame(points))
if pending:
    display(Markdown(pending))
if notes:
    display(Markdown("\n".join(f"- {note}" for note in notes)))
if False:
    display(Markdown(final_summary(data_dir, markdown_links=False)))

## Xem cấu hình và log theo fold

Hiển thị lượt có AUC cao nhất trong nhóm hiện tại.

In [ ]:
if rows:
    record = rows[0]
    directory = data_dir / record["name"]
    display(record["config"])
    display(pd.DataFrame([
        {key: item.get(key) for key in
         ("fold", "auc", "device", "seconds", "iterations", "converged", "rounds")}
        for item in record["folds"]
    ]))
    with np.load(directory / "predictions.npz", allow_pickle=False) as saved:
        display({name: saved[name].shape for name in ("ids", "fold", "oof", "test_ids", "test")})

## Phạm vi diễn giải

S71 đổi seed của cây, S93 đổi seed của bộ mã hóa. MLP là nhánh tùy chọn và không thuộc danh sách thí nghiệm của trình chạy này.